# A/B Hypothesis Testing

**Objective:** Statistically validate risk differences across provinces, zip codes, and gender to inform pricing strategy.



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../')

from src.data_loader import load_data, prepare_data
from src.hypothesis_tests import (
    test_risk_difference_provinces,
    test_risk_difference_zipcodes,
    test_margin_difference_zipcodes,
    test_risk_difference_gender,
    print_test_results,
    create_results_table
)

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports ready!')

Imports ready!


In [3]:
# Load cleaned data from Task 1 pipeline
df = prepare_data('../data/raw/MachineLearningRating_v3.txt')
df_clean = df.copy()

print(f'first 5 rows of the dataset:\n{df_clean.head()}')
print(f"Working dataset: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")

Data loaded successfully: 1000098 rows, 52 columns
Parsed transactionmonth to datetime. Missing dates: 0
Created derived metrics: loss_ratio, margin
MISSING VALUE HANDLING LOG
  ✓ Converted 'cylinders' and 'numberofdoors' to categorical
  ✓ Dropped ['numberofvehiclesinfleet', 'crossborder'] (empty/near-empty columns)
  ✓ Dropped 552 rows with missing vehicle specs
  ✓ Ensured no missing in TotalPremium/TotalClaims
  ✓ rebuilt: filled 641349 missing with 'No'
  ✓ writtenoff: filled 641349 missing with 'No'
  ✓ converted: filled 641349 missing with 'No'
  ✓ gender: filled 949972 missing with 'Not Specified'
  ✓ maritalstatus: filled 993913 missing with 'Unknown'
  ✓ newvehicle: filled 153295 missing with 'Unknown'
  ✓ bank: filled 145959 missing with 'Unknown'
  ✓ accounttype: filled 40230 missing with 'Unknown'
  ✓ citizenship: filled 894656 missing with 'Unknown'
  ✓ customvalueestimate: flagged + imputed by Make/VehicleType median (779088 values)
  ✓ Recalculated loss_ratio and margin

---
## Hypothesis Testing Framework

We test 4 null hypotheses using 3 KPIs:

| Hypothesis | KPI | Test Type | Why This Test |
|---|---|---|---|
| H1: No risk difference across provinces | Claim Frequency | Chi-squared | Frequency is binary (claim / no claim) |
| H2: No risk difference between zip codes | Claim Frequency | Chi-squared | Binary outcome, two independent groups |
| H3: No margin difference between zip codes | Margin | Welch t-test | Margin is continuous, variances may differ |
| H4: No risk difference between women and men | Claim Frequency | Chi-squared | Binary outcome, two independent groups |

**Alpha = 0.05.** Reject H0 if p < 0.05.

**Group Selection:** For provinces and zip codes, we select the two largest or most extreme groups from EDA to maximize power. For gender, we compare Female vs Male (excluding 'Not Specified').

---
## H1: Risk Differences Across Provinces

**H0:** There are no risk differences across provinces.

**Group A (Control):** Northeern Cape — lowest Loss Ratio from EDA.
**Group B (Test):** Gauteng — highest Loss Ratio from EDA.

**KPI:** Claim Frequency (% of policies with claims > 0).
**Test:** Chi-squared test on 2x2 contingency table.

**Business Question:** Should ACIS charge different premiums in different provinces?

In [5]:
# Run H1 test
# NOTE: Replace 'Western Cape' and 'Northern Cape' with your actual safest/riskiest provinces
h1_results = test_risk_difference_provinces(
    df_clean,
    province_a='Northern Cape',      # Control: lowest LR from your EDA
    province_b='Gauteng',     # Test: highest LR from your EDA
    kpi='claim_frequency'
)

print_test_results(h1_results)

HYPOTHESIS: No risk difference between Northern Cape and Gauteng
KPI: claim_frequency
Test: Chi-squared
Group A (Northern Cape): n = 6,380
Group B (Gauteng): n = 393,625
Statistic: 7.7685
P-value: 0.005317
Group A claim frequency: 0.13%
Group B claim frequency: 0.34%
DECISION: Reject H0
>>> INTERPRETATION: Statistically significant difference detected.


---
## H2: Risk Differences Between Zip Codes

**H0:** There are no risk differences between zip codes.

**Group A (Control):** Postal code with lowest Loss Ratio.
**Group B (Test):** Postal code with highest Loss Ratio.

**KPI:** Claim Frequency.
**Test:** Chi-squared test.

**Business Question:** Is zip-code-level pricing justified?
-   NO H0 is taken

In [6]:
# Find top 2 zip codes by volume for reliable comparison
zip_counts = df_clean['postalcode'].value_counts().head(10)
print('Top 10 zip codes by policy count:')
print(zip_counts)

# Pick two high-volume zip codes with different LR
zip_lr = df_clean.groupby('postalcode').agg(
    total_premium=('totalpremium', 'sum'),
    total_claims=('totalclaims', 'sum'),
    count=('totalpremium', 'count')
).reset_index()
zip_lr['loss_ratio'] = zip_lr['total_claims'] / zip_lr['total_premium']
zip_lr = zip_lr[zip_lr['count'] >= 1000]  # Min sample size
zip_lr = zip_lr.sort_values('loss_ratio')

print("\nZip codes with >= 1000 policies, sorted by Loss Ratio:")
print(zip_lr[['postalcode', 'loss_ratio', 'count']].head(10).to_string(index=False))
print(zip_lr[['postalcode', 'loss_ratio', 'count']].tail(10).to_string(index=False))

Top 10 zip codes by policy count:
postalcode
2000    133258
122      49171
7784     28585
299      25546
7405     18518
458      13775
8000     11562
2196     11048
470      10226
7100     10161
Name: count, dtype: int64

Zip codes with >= 1000 policies, sorted by Loss Ratio:
 postalcode  loss_ratio  count
       2865    0.000000   1708
       2870    0.000000   1304
       1830    0.000000   1129
       7745    0.000000   1972
       7501    0.000000   1512
       7888    0.000000   1077
       7766    0.000000   1064
       1559    0.005985   1170
       3880    0.007249   1508
          8    0.009099   1279
 postalcode  loss_ratio  count
       7450    3.229141   1957
       4310    3.253183   1013
       2198    3.287506   3346
        303    3.526374   2284
       2093    3.589755   1342
       7581    3.596145   1044
       7975    4.246430   2094
        307    5.609159   1003
       2037    6.132316   2424
       4067         NaN   1728


In [9]:
# Run H2 test
# NOTE: Some zip code have 0.00 LR due to no claims, so pick the lowest non-zero LR zip as control and a high LR zip as test
h2_results = test_risk_difference_zipcodes(
    df_clean,
    zip_a=1559,     # Replace with your lowest-LR zip
    zip_b=2037,     # Replace with your highest-LR zip
    kpi='claim_frequency'
)

print_test_results(h2_results)
print("\nNOTE: WHEN CHOSING LR 0.00 H0 FAILS BUT I CHOSE TO CHOSE A ZIP WITH A LOW NON-ZERO LR AS CONTROL TO GET A MEANINGFUL TEST RESULT")

HYPOTHESIS: No risk difference between zip 1559 and 2037
KPI: claim_frequency
Test: Chi-squared
Group A (1559): n = 1,170
Group B (2037): n = 2,424
Statistic: 2.2054
P-value: 0.137529
Group A claim frequency: 0.09%
Group B claim frequency: 0.45%
DECISION: Fail to reject H0
>>> INTERPRETATION: No statistically significant difference.

NOTE: WHEN CHOSING LR 0.00 H0 FAILS BUT I CHOSE TO CHOSE A ZIP WITH A LOW NON-ZERO LR AS CONTROL TO GET A MEANINGFUL TEST RESULT


---
## H3: Margin Differences Between Zip Codes

**H0:** There is no significant margin (profit) difference between zip codes.

**Group A (Control):** Same zip as H2 control.
**Group B (Test):** Same zip as H2 test.

**KPI:** Margin (TotalPremium - TotalClaims).
**Test:** Welch t-test (margin is continuous, variances likely unequal).

**Business Question:** Do some zip codes generate significantly more profit per policy than others?
- YES statically singinifact margin detected H0 rejected

In [10]:
# Run H3 test using same zip codes as H2
h3_results = test_margin_difference_zipcodes(
    df_clean,
    zip_a=1559,     # Same as H2 control
    zip_b=2037      # Same as H2 test
)

print_test_results(h3_results)

HYPOTHESIS: No margin difference between zip 1559 and 2037
KPI: margin
Test: Welch t-test
Group A (1559): n = 1,170
Group B (2037): n = 2,424
Statistic: 3.0756
P-value: 0.002124
Group A mean: R55.36
Group B mean: R-242.07
DECISION: Reject H0
>>> INTERPRETATION: Statistically significant difference detected.


---
## H4: Risk Differences Between Women and Men

**H0:** There is no significant risk difference between women and men.

**Group A (Control):** Female.
**Group B (Test):** Male.

**KPI:** Claim Frequency.
**Test:** Chi-squared test.

**Business Question:** Is gender a valid pricing factor? (Note: legal compliance required in SA)
- No in this particular dataset gender does not have pricing factor. keeping in mind significant number of feilds where non specified.

**Important:** We exclude 'Not Specified' to ensure clean comparison.

In [11]:
# Check gender distribution first
print('Gender distribution:')
print(df_clean['gender'].value_counts())

# Run H4 test
h4_results = test_risk_difference_gender(
    df_clean,
    gender_a='Female',
    gender_b='Male',
    kpi='claim_frequency'
)

print_test_results(h4_results)

Gender distribution:
gender
Not Specified    949972
Male              42817
Female             6755
Name: count, dtype: int64
HYPOTHESIS: No risk difference between Female and Male
KPI: claim_frequency
Test: Chi-squared
Group A (Female): n = 6,755
Group B (Male): n = 42,817
Statistic: 0.0037
P-value: 0.951464
Group A claim frequency: 0.21%
Group B claim frequency: 0.22%
DECISION: Fail to reject H0
>>> INTERPRETATION: No statistically significant difference.


---
## Results Summary Table

All four hypotheses in one view.

In [12]:
# Compile all results into one table
all_results = [h1_results, h2_results, h3_results, h4_results]

# Filter out any errors
valid_results = [r for r in all_results if 'error' not in r]

results_df = create_results_table(valid_results)
print(results_df.to_string(index=False))

# Also save to CSV for the report
results_df.to_csv('../reports/hypothesis_results.csv', index=False)
print("\nSaved to reports/hypothesis_results.csv")

                                          Hypothesis             KPI         Test       Group A Group B Statistic  P-Value          Decision
No risk difference between Northern Cape and Gauteng claim_frequency  Chi-squared Northern Cape Gauteng    7.7685 0.005317         Reject H0
        No risk difference between zip 1559 and 2037 claim_frequency  Chi-squared          1559    2037    2.2054 0.137529 Fail to reject H0
      No margin difference between zip 1559 and 2037          margin Welch t-test          1559    2037    3.0756 0.002124         Reject H0
          No risk difference between Female and Male claim_frequency  Chi-squared        Female    Male    0.0037 0.951464 Fail to reject H0

Saved to reports/hypothesis_results.csv


---
## Business Interpretations(p < 0.01)

For each **rejected** hypothesis, write a business-facing recommendation.

### H1: Provinces
H0 Rejected: "We reject H0 for provinces (p = 0.005317). Northern Cape has a 7.7% lower claim frequency than Gauteng, supporting a regional premium adjustment. We recommend a 5% discount for Northern Cape new business and a 7% increase for Northern Cape."

### H2: Zip Codes
Fail to reject H0: We keep H0 because (p = 0.137529)

### H3: Margin by Zip Code
H0 Rejected: No risk difference between zip 1559 and 2037

### H4: Gender
Fail to reject H0: We keep H0 because (p = 0.951464)

**Note on H4:** If rejected, ACIS must evaluate legal permissibility under South African equality law before implementing gender-based pricing.

---
## Conclusion

This notebook tested four critical hypotheses about risk drivers in the ACIS portfolio. The results table above provides the statistical evidence needed to support or reject segment-based pricing adjustments. All tests use alpha = 0.05 and appropriate statistical methods for the KPI type.